In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import RandomForestClassifier
warnings.simplefilter('ignore', FutureWarning)

# 1. データの読み込み
train_data = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

# 2. 空欄の穴埋め
num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_features:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(test_data[col].median())

cat_features = ['CryoSleep', 'VIP', 'HomePlanet', 'Destination']
for col in cat_features:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

# 3. 数値化
features = num_features + cat_features
X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
y_train = train_data['Transported'].astype(int)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 4. AIの学習と予測
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# 5. 提出用ファイルの出力
submission_preds = predictions.astype(bool)
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Transported': submission_preds})
output.to_csv('submission.csv', index=False)
print("【成功】準備完了！")


【成功】準備完了！


In [2]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# 1. 試してみたいパラメータの候補を辞書型で指定する
# （木の数を 100本 vs 200本、深さを 3 vs 5 vs 10 で総当たり戦をさせる）
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 10]
}

# 2. グリッドサーチのセッティング
# cv=5 はデータを5分割して、交代でテストしながら精度を測る設定（交差検証）
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=1),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1 # パソコンの頭脳をフルフルで使って高速化
)

print("最適なパラメータを探索中（総当たり戦を実行中）...")
grid_search.fit(X_train, y_train)

# 3. 一番結果が良かった組み合わせとスコアを表示
print("\n--- 探索結果 ---")
print(f"ベストパラメータ: {grid_search.best_params_}")
print(f"その時の内部スコア: {grid_search.best_score_:.4f}")

# 4. 一番良かったモデルをそのまま使ってテストデータを予測
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

# 5. 提出用CSVファイルを出力
submission_preds = predictions.astype(bool)
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Transported': submission_preds})
output.to_csv('submission.csv', index=False)
print("\n新しい submission.csv の出力が完了しました！")

最適なパラメータを探索中（総当たり戦を実行中）...

--- 探索結果 ---
ベストパラメータ: {'max_depth': 10, 'n_estimators': 100}
その時の内部スコア: 0.7958

新しい submission.csv の出力が完了しました！


In [3]:
# --- 特徴量エンジニアリング（客室番号の分解） ---

# 穴埋め用の文字として、一番多いパターン（最頻値）をあらかじめ取得
cabin_mode = train_data['Cabin'].mode()[0]

# 訓練データとテストデータの両方で、Cabinの空欄を埋めてから分解する
for df in [train_data, test_data]:
    # 1. まずCabinの空欄（NaN）を最頻値で埋める
    df['Cabin'] = df['Cabin'].fillna(cabin_mode)
    
    # 2. スラッシュで分解して、1番目（デッキ）と3番目（側）を新しい列にする
    df['Cabin_deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Cabin_side'] = df['Cabin'].apply(lambda x: x.split('/')[2])

print("客室番号の分解が完了しました！")
print("新しくできたデッキの種類:", train_data['Cabin_deck'].unique())
print("新しくできた側の種類:", train_data['Cabin_side'].unique())

客室番号の分解が完了しました！
新しくできたデッキの種類: ['B' 'F' 'A' 'G' 'E' 'D' 'C' 'T']
新しくできた側の種類: ['P' 'S']


In [4]:
# --- 新しい手掛かりを含めてAIに学習・予測させる ---

# 1. 手掛かり（features）のリストに、新しく作った2つの項目を追加する
features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 
            'CryoSleep', 'VIP', 'HomePlanet', 'Destination', 
            'Cabin_deck', 'Cabin_side'] # ← ここを追加！

# 2. pd.get_dummies で文字データを一気に 0 と 1 に数値化
X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
y_train = train_data['Transported'].astype(int)

# 3. 訓練データとテストデータの列をカチッと揃える
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 4. さっきのベストパラメータを設定したランダムフォレストを召喚
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=1)

# 5. 学習と予測を実行
print("新しい手掛かりを使ってAIが学習中...")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# 6. 提出用CSVファイルを出力
submission_preds = predictions.astype(bool)
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Transported': submission_preds})
output.to_csv('submission.csv', index=False)

print("\n新しい武器を装備した submission.csv の出力が完了しました！")

新しい手掛かりを使ってAIが学習中...

新しい武器を装備した submission.csv の出力が完了しました！
